# Pipeline Inspection: Dataset Pipeline (DLR-TomoSAR)

This notebook inspects the **dataset construction pipeline** orchestrated by
`pipelines.dataset_pipeline.pipeline.DatasetPipeline.run()` for tomographic SAR
reconstruction. The pipeline turns a co-registered stack of complex SAR acquisitions
(one primary pass, several secondary passes, their interferograms, and an optional DEM)
plus a per-pixel Gaussian-mixture parameter cube into normalized, patched, augmented
`(input_tensor, gt_parameters)` pairs wrapped in train / validation / test
`DataLoader`s.

`DatasetPipeline.run()` sequences: **build the train `PatchDataset`** (Layout → Cropper
→ Patcher → PatchDataset) → **fit normalization statistics on the train split only**
(`StatsComputer.compute`) → **wrap those statistics in a `Normalizer`** and attach it to
the train dataset → **build the val and test `PatchDataset`s sharing the train-fitted
normalizer** → **assemble the three `DataLoader`s** (`Loader.build`) → **persist
dataset / crop / patch metadata** (`MetadataWriter`). Inside every dataset
`__getitem__`, the per-patch transform composes: complex-patch extraction (with
reflective padding at borders) → complex→real channel conversion via `Representation`
→ ground-truth channel selection → train-only spatial augmentation → per-channel
normalization (with optional `log1p`).

This notebook verifies each of those sub-steps in isolation, with layered structural,
statistical, and semantic assertions, so behavioural equivalence can be demonstrated
before and after any refactor.

**Note on data availability.** The underlying SAR tomogram `.npy` arrays, the
pre-processing run directory, and some external repositories are not present in this
environment. Each stage therefore exercises the *real* pipeline component classes
(`Layout`, `Cropper`, `Patcher`, `PatchDataset`, `StatsComputer`, `Normalizer`,
`SpatialAugmenter`, `Loader`, `MetadataWriter`) against config-derived synthetic
memory-mapped arrays that match the production shapes, dtypes, and value ranges. A
self-contained on-disk fixture (a fake pre-processing run) is materialized in Stage 0
so that the genuine `Layout`/`Cropper` file-reading code paths execute unchanged. The
notebook is executable top-to-bottom once the real pre-processing run is mounted:
replace `PREPROCESSING_RUN_DIRECTORY` with the real path and skip the fixture cell.

In [ ]:
from __future__ import annotations

import os
import sys
import json
import math
import shutil
import tempfile
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS",      "1")
os.environ.setdefault("MKL_NUM_THREADS",      "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import numpy as np
import torch
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({
    "font.family":        "serif",
    "font.size":          11,
    "axes.labelsize":     12,
    "axes.titlesize":     13,
    "legend.fontsize":    10,
    "xtick.labelsize":    10,
    "ytick.labelsize":    10,
    "figure.dpi":         150,
    "savefig.dpi":        300,
    "savefig.bbox":       "tight",
    "axes.spines.top":    False,
    "axes.spines.right":  False,
})

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pipelines").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

FIGURE_DIRECTORY = PROJECT_ROOT / "notebooks" / "figures" / "inspect_dataset_pipeline"
FIGURE_DIRECTORY.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 0
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

PALETTE = {
    "primary":   "#1f4e79",
    "secondary": "#8c6d31",
    "accent":    "#7a3b2e",
    "neutral":   "#4d4d4d",
    "before":    "#9aa7b5",
    "after":     "#1f4e79",
}

from tools.crop_region                         import CropRegion
from tools.split_regions                       import SplitRegions
from tools.representation                      import Representation
from tools.logger                              import Logger
from configuration.norm_config                 import NormMethod, ChannelStrategy
from configuration.dataset_config              import (
    InputConfig, OutputConfig, PatchConfiguration, AugmentationConfig, DatasetConfiguration,
)
from pipelines.dataset_pipeline.layout         import Layout
from pipelines.dataset_pipeline.crop           import Cropper
from pipelines.dataset_pipeline.patch          import Patcher, GridInfo
from pipelines.dataset_pipeline.dataset        import PatchDataset
from pipelines.dataset_pipeline.stats          import Stats
from pipelines.dataset_pipeline.stats_computer import StatsComputer
from pipelines.dataset_pipeline.normalizer     import Normalizer
from pipelines.dataset_pipeline.augmentation   import SpatialAugmenter
from pipelines.dataset_pipeline.loader         import Loader
from pipelines.dataset_pipeline.metadata       import MetadataWriter
from pipelines.dataset_pipeline.pipeline       import DatasetPipeline


## Pipeline Overview

```
SAR stack on disk  (primary + secondaries + interferograms + DEM)  +  GT parameter cube (3K, H, W)
  └─► Stage 1:  Layout.__init__           →  global CropRegion, artifact paths, tags  (reads dataset.json)
        └─► Stage 2:  Cropper.load_split  →  per-split stacked inputs (P,az,rg) + dem + parameters (3K,az,rg)
              └─► Stage 3:  Patcher.build  →  GridInfo: n_v·n_h patch coords + reflective pad budget
                    └─► Stage 4:  PatchDataset.__getitem__  →  (input_tensor C×ph×pw, gt_params 3K'×ph×pw)
                          └─► Stage 5:  StatsComputer.compute (TRAIN ONLY)  →  per-channel loc/scale + log1p flags
                                └─► Stage 6:  Normalizer.normalize_*  →  standardized tensors, log1p magnitudes
                                      └─► Stage 7:  SpatialAugmenter  →  flip / rot90 / noise (train split only)
                                            └─► Stage 8:  build val + test PatchDataset (shared normalizer)
                                                  └─► Stage 9:  Loader.build  →  train/val/test DataLoaders
                                                        └─► Stage 10: MetadataWriter  →  crop.json, patch.json, config.json
```

| Stage | Callable | Input | Output |
|-------|----------|-------|--------|
| 1  | `Layout(run_directory, logger, parameters_path)` | pre-processing run dir + `parameters_path` | `global_crop : CropRegion`, `artifacts : dict`, tomogram/parameter tags |
| 2  | `Cropper(layout, split_regions, logger).load_split(region)` | a split `CropRegion` | `{"inputs": (P,az,rg), "dem": (az,rg), "parameters": (3K,az,rg)}` |
| 3  | `Patcher.build(spatial_size, patch_size, stride, use_reflective_padding)` | split `(az,rg)` + patch cfg | `Patcher` holding `GridInfo` + per-patch crop/pad coords |
| 4  | `PatchDataset(...).__getitem__(idx)` | one patch index | `(input_tensor float32 C×ph×pw, gt_params float32 3K'×ph×pw)` |
| 5  | `StatsComputer.compute(dataset=train_ds, ...)` | **train** dataset + GT params path | `Stats` with per-channel `loc`, `scale`, strategy, `log1p` |
| 6  | `Normalizer(stats).normalize_input / normalize_output` | raw patch tensors | standardized tensors (≈ zero-centred) |
| 7  | `SpatialAugmenter(aug_cfg, logger)(input, gt)` | one `(input, gt)` pair (train) | spatially transformed + noised pair |
| 8  | `DatasetPipeline._build_dataset("val"/"test", normalizer)` | val / test region | `PatchDataset` reusing the **train** normalizer |
| 9  | `Loader.build(train, val, test, batch_size, ...)` | the three datasets | three `DataLoader`s (train shuffled + `drop_last`) |
| 10 | `MetadataWriter.save_*` | config, crop, grids | `dataset_creation_config.json`, `crop.json`, `patch.json` |

The remaining cells build the synthetic on-disk fixture and the configuration objects
every stage consumes, then walk the ten stages in order.

In [ ]:
class SyntheticPreprocessingRun:
    def __init__(self, root_directory: Path, azimuth_size: int, range_size: int, n_secondaries: int, n_gaussians: int, seed: int) -> None:
        self.root_directory = Path(root_directory)
        self.azimuth_size   = azimuth_size
        self.range_size     = range_size
        self.n_secondaries  = n_secondaries
        self.n_gaussians    = n_gaussians
        self.rng            = np.random.default_rng(seed)
        self.data_directory = self.root_directory / "data"

    def _complex_field(self, n_passes: int) -> np.ndarray:
        magnitude = self.rng.lognormal(mean=0.0, sigma=0.8, size=(n_passes, self.azimuth_size, self.range_size)).astype(np.float32)
        phase     = self.rng.uniform(-np.pi, np.pi, size=(n_passes, self.azimuth_size, self.range_size)).astype(np.float32)
        return (magnitude * np.exp(1j * phase)).astype(np.complex64)

    def _parameter_cube(self) -> np.ndarray:
        cube = np.empty((3 * self.n_gaussians, self.azimuth_size, self.range_size), dtype=np.float32)
        for gaussian_index in range(self.n_gaussians):
            cube[gaussian_index * 3 + 0] = self.rng.lognormal(mean=-0.5, sigma=0.6, size=(self.azimuth_size, self.range_size)).astype(np.float32)
            cube[gaussian_index * 3 + 1] = self.rng.uniform(-30.0, 30.0, size=(self.azimuth_size, self.range_size)).astype(np.float32)
            cube[gaussian_index * 3 + 2] = self.rng.uniform(0.5, 8.0, size=(self.azimuth_size, self.range_size)).astype(np.float32)
        return cube

    def _digital_elevation_model(self) -> np.ndarray:
        return self.rng.normal(loc=120.0, scale=15.0, size=(self.azimuth_size, self.range_size)).astype(np.float32)

    def materialize(self, global_crop: CropRegion) -> dict:
        self.data_directory.mkdir(parents=True, exist_ok=True)

        primary        = self._complex_field(1)
        secondaries    = self._complex_field(self.n_secondaries)
        interferograms = self._complex_field(self.n_secondaries)
        parameters     = self._parameter_cube()
        elevation      = self._digital_elevation_model()

        np.save(self.data_directory / "primary_reduced.npy",        primary)
        np.save(self.data_directory / "secondaries_reduced.npy",    secondaries)
        np.save(self.data_directory / "interferograms_reduced.npy", interferograms)
        np.save(self.data_directory / "dem_reduced.npy",            elevation)

        parameters_path = self.data_directory / "parameters.npy"
        np.save(parameters_path, parameters)

        artifacts = {
            "primary_reduced":        "primary_reduced.npy",
            "secondaries_reduced":    "secondaries_reduced.npy",
            "interferograms_reduced": "interferograms_reduced.npy",
            "dem_reduced":            "dem_reduced.npy",
        }

        layout_payload = {
            "global_crop":   list(global_crop.as_tuple()),
            "dataset_type":  "synthetic_tomosar",
            "tomogram_tag":  "synthetic",
            "parameter_tag": "synthetic_gmm",
            "artifacts":     artifacts,
        }
        with open(self.data_directory / "dataset.json", "w", encoding="utf-8") as handle:
            json.dump(layout_payload, handle, indent=4)

        return {"parameters_path": parameters_path, "artifacts": artifacts}


AZIMUTH_SIZE   = 256
RANGE_SIZE     = 192
N_SECONDARIES  = 4
N_GAUSSIANS    = 1

GLOBAL_CROP = CropRegion(azimuth_start=0, azimuth_end=AZIMUTH_SIZE, range_start=0, range_end=RANGE_SIZE)

WORKING_RUN_ROOT            = Path(tempfile.mkdtemp(prefix="tomosar_dataset_inspect_"))
PREPROCESSING_RUN_DIRECTORY = WORKING_RUN_ROOT / "preprocessing_run"
TRAINING_RUN_DIRECTORY      = WORKING_RUN_ROOT / "training_run"

fixture        = SyntheticPreprocessingRun(PREPROCESSING_RUN_DIRECTORY, AZIMUTH_SIZE, RANGE_SIZE, N_SECONDARIES, N_GAUSSIANS, RANDOM_SEED)
fixture_handle = fixture.materialize(GLOBAL_CROP)
PARAMETERS_PATH = fixture_handle["parameters_path"]

print("Synthetic pre-processing run materialized at:", PREPROCESSING_RUN_DIRECTORY)
print("  global crop (az_start, az_end, rg_start, rg_end):", GLOBAL_CROP.as_tuple())
print("  secondaries:", N_SECONDARIES, " interferograms:", N_SECONDARIES, " gaussians:", N_GAUSSIANS)
print("  parameter cube path:", PARAMETERS_PATH.name, "with", 3 * N_GAUSSIANS, "channels")


In [ ]:
SPLIT_REGIONS = SplitRegions.from_ratios(GLOBAL_CROP, train_ratio=0.70, val_ratio=0.15)

INPUT_CONFIG = InputConfig(
    use_primary                   = True,
    primary_representation        = Representation.MAG_ONLY,
    use_secondaries               = True,
    secondaries_representation    = Representation.MAG_ONLY,
    use_interferograms            = True,
    interferograms_representation = Representation.ANGLE_ONLY,
    use_dem                       = False,
)

OUTPUT_CONFIG = OutputConfig(use_amplitude=True, use_mu=True, use_sigma=True)

PATCH_CONFIG = PatchConfiguration(size=(64, 64), stride=32, use_reflective_padding=True)

AUGMENTATION_CONFIG = AugmentationConfig(
    p_flip_h  = 0.5,
    p_flip_v  = 0.5,
    p_rot90   = 0.5,
    noise_std = 0.01,
    p_noise   = 1.0,
)

SPECTRAL_X_AXIS = np.linspace(-30.0, 30.0, 64, dtype=np.float32)

DATASET_CONFIG = DatasetConfiguration(
    preprocessing_run_directory = PREPROCESSING_RUN_DIRECTORY,
    split_regions               = SPLIT_REGIONS,
    parameters_path             = PARAMETERS_PATH,
    patch                       = PATCH_CONFIG,
    input_config                = INPUT_CONFIG,
    output_config               = OUTPUT_CONFIG,
    augmentation                = AUGMENTATION_CONFIG,
    batch_size                  = 8,
    num_workers                 = 0,
    shuffle_train               = True,
    pin_memory                  = False,
    x_axis                      = SPECTRAL_X_AXIS,
    n_gaussians                 = N_GAUSSIANS,
)

TRAINING_RUN_DIRECTORY.mkdir(parents=True, exist_ok=True)
INSPECTION_LOGGER = Logger(log_dir=str(TRAINING_RUN_DIRECTORY / "logs"), name="dataset_inspect", level="INFO")

PATCH_HEIGHT, PATCH_WIDTH = PATCH_CONFIG.size
PATCH_STRIDE              = PATCH_CONFIG.stride

print("Split azimuth extents (start, end):")
for split_name, region in SPLIT_REGIONS.items():
    print(f"  {split_name:<5}: {region.azimuth_start:>4} .. {region.azimuth_end:<4}  (az_size={region.azimuth_size}, rg_size={region.range_size})")
print()
print("Patch size:", PATCH_CONFIG.size, " stride:", PATCH_STRIDE, " reflective padding:", PATCH_CONFIG.use_reflective_padding)
print("Output role names:", OUTPUT_CONFIG.role_names, " params/gaussian:", OUTPUT_CONFIG.params_per_gaussian)


**Passing to Stage 1:** the on-disk synthetic pre-processing run, `DATASET_CONFIG` (a real `DatasetConfiguration`), and `INSPECTION_LOGGER`. Every subsequent stage consumes config fields rather than literals.

---
## Stage 1: Layout — resolve global crop and artifact paths

**Callable:** `Layout(run_directory, logger, parameters_path)`
**Input:** the pre-processing run directory (containing `data/dataset.json`) and the GT `parameters_path`.
**Output:** a `Layout` exposing `global_crop : CropRegion`, the `artifacts` map (artifact-key → relative `.npy` name), the tomogram / parameter tags, and `artifact_path(key)` resolution.

The layout reads the JSON manifest written by the pre-processing pipeline and reconstructs the global acquisition window:

$$
\text{global\_crop} = (\text{az}_{\text{start}}, \text{az}_{\text{end}}, \text{rg}_{\text{start}}, \text{rg}_{\text{end}})
$$

> **What you should see:** `global_crop.azimuth_size` equals the configured azimuth
> extent (`AZIMUTH_SIZE` = 256) and `range_size` equals `RANGE_SIZE` = 192. The
> `artifacts` dict contains the five expected keys (`primary_reduced`,
> `secondaries_reduced`, `interferograms_reduced`, `dem_reduced`, plus the special
> `parameters` key routed to `parameters_path`). Each resolved `artifact_path` must
> point at an existing `.npy` file.

In [ ]:
layout = Layout(
    run_directory   = DATASET_CONFIG.preprocessing_run_directory,
    logger          = INSPECTION_LOGGER,
    parameters_path = DATASET_CONFIG.parameters_path,
)


In [ ]:
print("Layout global crop tuple         :", layout.global_crop.as_tuple())
print("  azimuth size (lines)           :", layout.global_crop.azimuth_size)
print("  range size   (samples)         :", layout.global_crop.range_size)
print("Dataset type                     :", layout.dataset_type)
print("Tomogram tag                     :", layout.tomogram_tag, " parameter tag:", layout.parameter_tag)
print()
print("Artifact keys                    :", sorted(layout.artifacts.keys()))
print()
for artifact_key in ["primary_reduced", "secondaries_reduced", "interferograms_reduced", "dem_reduced", "parameters"]:
    resolved_path = layout.artifact_path(artifact_key)
    print(f"  artifact_path({artifact_key!r:>24}) -> exists={resolved_path.exists()}  {resolved_path.name}")


In [ ]:
artifact_keys      = ["primary_reduced", "secondaries_reduced", "interferograms_reduced", "dem_reduced", "parameters"]
artifact_megabytes = [layout.artifact_path(key).stat().st_size / 1e6 for key in artifact_keys]

figure, axes = plt.subplots(1, 2, figsize=(10, 3.8))

axes[0].bar(range(len(artifact_keys)), artifact_megabytes, color=PALETTE["primary"], edgecolor="white", linewidth=0.6)
axes[0].set_xticks(range(len(artifact_keys)))
axes[0].set_xticklabels([key.replace("_reduced", "") for key in artifact_keys], rotation=30, ha="right")
axes[0].set_title("Stage 1 — Layout: artifact sizes on disk")
axes[0].set_xlabel("artifact key")
axes[0].set_ylabel("file size (MB)")

global_box = layout.global_crop
axes[1].add_patch(plt.Rectangle((global_box.range_start, global_box.azimuth_start), global_box.range_size, global_box.azimuth_size, fill=False, edgecolor=PALETTE["accent"], linewidth=1.8))
axes[1].set_xlim(global_box.range_start - 10, global_box.range_end + 10)
axes[1].set_ylim(global_box.azimuth_end + 10, global_box.azimuth_start - 10)
axes[1].set_title("Stage 1 — Layout: global crop window")
axes[1].set_xlabel("range (samples)")
axes[1].set_ylabel("azimuth (lines)")

figure.tight_layout()
figure.savefig(FIGURE_DIRECTORY / "stage_01_layout.pdf")
plt.show()
print("Saved figure -> notebooks/figures/inspect_dataset_pipeline/stage_01_layout.pdf")


In [ ]:
expected_artifact_keys = {"primary_reduced", "secondaries_reduced", "interferograms_reduced", "dem_reduced"}

checks = [
    ("Layout azimuth size matches global crop config", layout.global_crop.azimuth_size == GLOBAL_CROP.azimuth_size),
    ("Layout range size matches global crop config",   layout.global_crop.range_size   == GLOBAL_CROP.range_size),
    ("All expected artifact keys present",             expected_artifact_keys.issubset(set(layout.artifacts.keys()))),
    ("parameters key routes to parameters_path",       layout.artifact_path("parameters") == Path(DATASET_CONFIG.parameters_path)),
    ("Every artifact path exists on disk",             all(layout.artifact_path(key).exists() for key in expected_artifact_keys)),
]
for description, condition in checks:
    print(f"[{'PASS' if condition else 'FAIL'}] {description}")


### Common Mistakes — Stage 1: Layout

| Symptom | Likely Cause | How to Diagnose |
|---------|-------------|-----------------|
| `FileNotFoundError` on `dataset.json` | `preprocessing_run_directory` points one level above/below the run; `data/` subdir missing | Print `layout.data_directory`; confirm `data/dataset.json` exists |
| `global_crop` size disagrees with arrays | `dataset.json` written by a different crop than the arrays were reduced with | Compare `layout.global_crop.as_tuple()` to `primary_reduced.shape[-2:]` |
| `parameters` artifact resolves into `data/` | `parameters_path` left `None`, so the special-case branch in `artifact_path` is skipped | Print `layout.artifact_path("parameters")`; it must equal `parameters_path` |
| `KeyError` resolving an artifact | artifact key renamed in pre-processing but not in this loader | Print `layout.artifacts.keys()` and match against requested keys |

**Passing to Stage 2:** `layout` — a `Layout` with a validated `global_crop` and artifact resolver, consumed by the `Cropper`.

---
## Stage 2: Cropper — load and stack a split into memory

**Callable:** `Cropper(layout, split_regions, logger).load_split(region)`
**Input:** a per-split `CropRegion` (here the **train** region).
**Output:** a dict `{"inputs": (P, az, rg), "dem": (az, rg), "parameters": (3K, az, rg)}` where the input stack interleaves `1` primary + `S` secondaries + `S` interferograms, so `P = 1 + 2S`.

The split crop is mapped into the global array's local coordinates:

$$
\text{az\_slice} = [\,r_{\text{az,start}} - g_{\text{az,start}} \;:\; r_{\text{az,end}} - g_{\text{az,start}}\,], \qquad
\text{rg\_slice} = [\,r_{\text{rg,start}} - g_{\text{rg,start}} \;:\; r_{\text{rg,end}} - g_{\text{rg,start}}\,]
$$

and the passes are stacked along a new leading axis:

$$
\text{inputs}[0] = \text{primary}, \quad
\text{inputs}[1 : 1+S] = \text{secondaries}, \quad
\text{inputs}[1+S :] = \text{interferograms}.
$$

> **What you should see:** for the train split, `inputs.shape == (1 + 2·N_SECONDARIES,
> train_az_size, RANGE_SIZE)` with complex dtype; `parameters.shape == (3·N_GAUSSIANS,
> train_az_size, RANGE_SIZE)` and `dem.shape == (train_az_size, RANGE_SIZE)` as
> `float32`. The train azimuth size must equal the train `CropRegion.azimuth_size`. No
> NaN / Inf in any array.

In [ ]:
cropper = Cropper(layout=layout, split_regions=DATASET_CONFIG.split_regions, logger=INSPECTION_LOGGER)

train_region = dict(DATASET_CONFIG.split_regions.items())["train"]
train_arrays = cropper.load_split(train_region)


In [ ]:
train_inputs     = train_arrays["inputs"]
train_dem        = train_arrays["dem"]
train_parameters = train_arrays["parameters"]

print("Train region tuple               :", train_region.as_tuple())
print("Local azimuth slice / range slice:", cropper.to_local_slices(train_region))
print()
print("inputs     shape / dtype         :", train_inputs.shape, train_inputs.dtype)
print("  expected passes P = 1 + 2S     :", 1 + 2 * N_SECONDARIES)
print("dem        shape / dtype         :", train_dem.shape, train_dem.dtype)
print("parameters shape / dtype         :", train_parameters.shape, train_parameters.dtype)
print()
print("inputs |mag| min / max / mean    :", float(np.abs(train_inputs).min()), float(np.abs(train_inputs).max()), float(np.abs(train_inputs).mean()))
print("dem            min / max / mean  :", float(train_dem.min()), float(train_dem.max()), float(train_dem.mean()))
print("parameters     min / max / mean  :", float(train_parameters.min()), float(train_parameters.max()), float(train_parameters.mean()))
print()
print("NaN in inputs / dem / parameters :", bool(np.isnan(train_inputs).any()), bool(np.isnan(train_dem).any()), bool(np.isnan(train_parameters).any()))


In [ ]:
pass_labels = ["primary"] + [f"secondary {index + 1}" for index in range(N_SECONDARIES)] + [f"interferogram {index + 1}" for index in range(N_SECONDARIES)]

figure, axes = plt.subplots(1, 3, figsize=(13, 4.0))

primary_magnitude = np.abs(train_inputs[0])
amplitude_image   = axes[0].imshow(primary_magnitude, aspect="auto", cmap="magma", origin="upper")
axes[0].set_title("Stage 2 — Cropper: primary magnitude")
axes[0].set_xlabel("range (samples)")
axes[0].set_ylabel("azimuth (lines)")
figure.colorbar(amplitude_image, ax=axes[0], fraction=0.046, label="|amplitude|")

interferogram_phase = np.angle(train_inputs[1 + N_SECONDARIES])
phase_image         = axes[1].imshow(interferogram_phase, aspect="auto", cmap="twilight", origin="upper", vmin=-np.pi, vmax=np.pi)
axes[1].set_title("Stage 2 — Cropper: interferogram phase")
axes[1].set_xlabel("range (samples)")
axes[1].set_ylabel("azimuth (lines)")
figure.colorbar(phase_image, ax=axes[1], fraction=0.046, label="phase (rad)")

per_pass_mean = np.abs(train_inputs).mean(axis=(1, 2))
axes[2].bar(range(len(per_pass_mean)), per_pass_mean, color=PALETTE["secondary"], edgecolor="white", linewidth=0.5)
axes[2].set_xticks(range(len(pass_labels)))
axes[2].set_xticklabels(pass_labels, rotation=45, ha="right", fontsize=7)
axes[2].set_title("Stage 2 — Cropper: mean magnitude per pass")
axes[2].set_xlabel("stacked pass")
axes[2].set_ylabel("mean |amplitude|")

figure.tight_layout()
figure.savefig(FIGURE_DIRECTORY / "stage_02_cropper.pdf")
plt.show()
print("Saved figure -> notebooks/figures/inspect_dataset_pipeline/stage_02_cropper.pdf")


In [ ]:
expected_passes     = 1 + 2 * N_SECONDARIES
train_azimuth_size  = train_region.azimuth_size

checks = [
    ("inputs leading dim equals 1 + 2S",            train_inputs.shape[0] == expected_passes),
    ("inputs azimuth equals train region azimuth",  train_inputs.shape[1] == train_azimuth_size),
    ("inputs range equals configured range size",   train_inputs.shape[2] == RANGE_SIZE),
    ("parameters channels equal 3K",                train_parameters.shape[0] == 3 * N_GAUSSIANS),
    ("dem spatial shape matches split",             train_dem.shape == (train_azimuth_size, RANGE_SIZE)),
    ("inputs are complex dtype",                    np.iscomplexobj(train_inputs)),
    ("no NaN / Inf in stacked inputs",              bool(np.isfinite(train_inputs).all())),
    ("no NaN / Inf in parameters",                  bool(np.isfinite(train_parameters).all())),
]
for description, condition in checks:
    print(f"[{'PASS' if condition else 'FAIL'}] {description}")


### Common Mistakes — Stage 2: Cropper

| Symptom | Likely Cause | How to Diagnose |
|---------|-------------|-----------------|
| Negative or out-of-bounds slice | split `CropRegion` not contained in `global_crop`; `to_local_slices` yields negative start | Print `cropper.to_local_slices(region)` and compare to `global_crop` |
| Splits overlap in azimuth (leakage) | `SplitRegions.from_ratios` boundaries miscomputed or regions hand-edited to overlap | Assert `train.azimuth_end == val.azimuth_start == ...`; check disjointness |
| Interferograms stacked before secondaries | wrong slice offsets in the `inputs_split` assembly | Compare `inputs[1:1+S]` vs `inputs[1+S:]` magnitude/phase signatures |
| `parameters` not contiguous → silent slow patching | `np.ascontiguousarray` skipped on the parameter view | Print `train_parameters.flags["C_CONTIGUOUS"]` |
| DEM dtype stays float64 | `.astype(np.float32)` dropped, doubling memory | Print `train_dem.dtype` |

**Passing to Stage 3:** `train_arrays` — stacked `inputs (P,az,rg)`, `dem`, `parameters (3K,az,rg)` for the train split; the spatial size `(az, rg)` drives patch-grid construction.

---
## Stage 3: Patcher — build the patch grid and padding budget

**Callable:** `Patcher.build(spatial_size, patch_size, stride, use_reflective_padding)`
**Input:** the split spatial size `(H, W) = (train_az, range)` and the patch config `(p_h, p_w)`, `stride`, padding flag.
**Output:** a `Patcher` holding a `GridInfo` (`n_v × n_h` patches, symmetric pad budget) plus a per-patch list of crop bounds and pad specs.

The number of patches along each axis and the padding required to make the last patch land exactly on the border:

$$
n_v = \left\lceil \frac{H - p_h}{s} \right\rceil + 1, \qquad
n_h = \left\lceil \frac{W - p_w}{s} \right\rceil + 1
$$
$$
\text{pad}_v = \big(p_h + (n_v - 1)\,s\big) - H, \qquad
\text{pad}_h = \big(p_w + (n_h - 1)\,s\big) - W
$$

split symmetrically into top/bottom and left/right. Reflective (`symmetric`) padding fills border patches; otherwise zeros.

> **What you should see:** `number_of_patches == n_v · n_h` and is strictly positive.
> The padded size `(H + pad_v, W + pad_h)` must satisfy `padded - p ≡ 0 (mod s)` on both
> axes, i.e. the patches tile the padded canvas with stride `s` and no remainder. Every
> per-patch crop bound lies within `[0, H] × [0, W]`; only border patches carry a
> non-`None` pad spec.

In [ ]:
train_spatial_size = (train_region.azimuth_size, train_region.range_size)

train_patcher = Patcher.build(
    spatial_size           = train_spatial_size,
    patch_size             = DATASET_CONFIG.patch.size,
    stride                 = DATASET_CONFIG.patch.stride,
    use_reflective_padding = DATASET_CONFIG.patch.use_reflective_padding,
)
train_grid = train_patcher.grid


In [ ]:
print("Spatial size (H, W)              :", train_grid.spatial_size)
print("Patch size (p_h, p_w)            :", train_grid.patch_size, " stride:", train_grid.stride)
print("Grid n_v x n_h                   :", train_grid.n_v, "x", train_grid.n_h, "=", train_grid.number_of_patches, "patches")
print()
print("Padding top / bottom             :", train_grid.pad_top, "/", train_grid.pad_bot, " (vertical total", train_grid.padding_vertical, ")")
print("Padding left / right             :", train_grid.pad_left, "/", train_grid.pad_right, " (horizontal total", train_grid.padding_horizontal, ")")
print("Padded size (H', W')             :", train_grid.padded_size)
print("Reflective padding               :", train_grid.use_reflective_padding)
print()
border_patches = sum(1 for coords in train_patcher._patch_coords if coords[4] is not None)
print("Patches requiring border padding :", border_patches, "/", train_grid.number_of_patches)
print("First patch crop bounds (v0,v1,h0,h1,pad):", train_patcher._patch_coords[0])
print("Last  patch crop bounds (v0,v1,h0,h1,pad):", train_patcher._patch_coords[-1])


In [ ]:
coverage = np.zeros(train_grid.spatial_size, dtype=np.int32)
for v0c, v1c, h0c, h1c, _ in train_patcher._patch_coords:
    coverage[v0c:v1c, h0c:h1c] += 1

figure, axes = plt.subplots(1, 2, figsize=(11, 4.2))

coverage_image = axes[0].imshow(coverage, aspect="auto", cmap="viridis", origin="upper")
axes[0].set_title("Stage 3 — Patcher: per-pixel patch coverage")
axes[0].set_xlabel("range (samples)")
axes[0].set_ylabel("azimuth (lines)")
figure.colorbar(coverage_image, ax=axes[0], fraction=0.046, label="patches covering pixel")

patch_height, patch_width = train_grid.patch_size
for vertical_index in range(train_grid.n_v):
    for horizontal_index in range(train_grid.n_h):
        top  = vertical_index   * train_grid.stride - train_grid.pad_top
        left = horizontal_index * train_grid.stride - train_grid.pad_left
        axes[1].add_patch(plt.Rectangle((left, top), patch_width, patch_height, fill=False, edgecolor=PALETTE["accent"], linewidth=0.6, alpha=0.7))
axes[1].add_patch(plt.Rectangle((0, 0), train_grid.spatial_size[1], train_grid.spatial_size[0], fill=False, edgecolor=PALETTE["primary"], linewidth=2.0))
axes[1].set_xlim(-train_grid.pad_left - 5, train_grid.spatial_size[1] + train_grid.pad_right + 5)
axes[1].set_ylim(train_grid.spatial_size[0] + train_grid.pad_bot + 5, -train_grid.pad_top - 5)
axes[1].set_title("Stage 3 — Patcher: patch tiling over padded canvas")
axes[1].set_xlabel("range (samples)")
axes[1].set_ylabel("azimuth (lines)")

figure.tight_layout()
figure.savefig(FIGURE_DIRECTORY / "stage_03_patcher.pdf")
plt.show()
print("Saved figure -> notebooks/figures/inspect_dataset_pipeline/stage_03_patcher.pdf")


In [ ]:
patch_height, patch_width = train_grid.patch_size
padded_height, padded_width = train_grid.padded_size

stride_tiles_vertical   = (padded_height - patch_height) % train_grid.stride == 0
stride_tiles_horizontal = (padded_width  - patch_width)  % train_grid.stride == 0
bounds_within_canvas    = all((0 <= v0 <= v1 <= train_grid.spatial_size[0]) and (0 <= h0 <= h1 <= train_grid.spatial_size[1]) for v0, v1, h0, h1, _ in train_patcher._patch_coords)
every_pixel_covered     = bool((coverage >= 1).all())

checks = [
    ("number_of_patches equals n_v * n_h",          train_grid.number_of_patches == train_grid.n_v * train_grid.n_h),
    ("number_of_patches is positive",               train_grid.number_of_patches > 0),
    ("patches tile padded height with stride",      stride_tiles_vertical),
    ("patches tile padded width with stride",       stride_tiles_horizontal),
    ("padded size >= spatial size on both axes",    padded_height >= train_grid.spatial_size[0] and padded_width >= train_grid.spatial_size[1]),
    ("all crop bounds within spatial canvas",       bounds_within_canvas),
    ("every interior pixel covered >= 1 patch",     every_pixel_covered),
    ("coord list length equals patch count",        len(train_patcher._patch_coords) == train_grid.number_of_patches),
]
for description, condition in checks:
    print(f"[{'PASS' if condition else 'FAIL'}] {description}")


### Common Mistakes — Stage 3: Patcher

| Symptom | Likely Cause | How to Diagnose |
|---------|-------------|-----------------|
| Off-by-one patch count | `math.ceil((H - p)/s) + 1` replaced with floor, or `+1` dropped | Recompute `n_v`/`n_h` by hand and compare to `GridInfo` |
| Stitched reconstruction shifted | asymmetric `pad_top`/`pad_left` ignored at inference; train and inference patch origins must match | Print `pad_top/pad_left`; confirm both use `pad_v // 2` |
| Border seams in stitched output | constant (zero) padding used where reflective was intended | Check `grid.use_reflective_padding`; inspect a border patch's filled region |
| Last patch reads out of bounds | crop bounds not clamped to `[0, H]` before padding | Verify `v1c = min(H, v1)` and pad spec carries the overflow |
| Coverage gaps when `stride > patch` | stride exceeds patch size, leaving uncovered strips | Inspect the coverage heatmap for zero pixels |

**Passing to Stage 4:** `train_patcher` — extracts patch `idx` from any `(..., H, W)` array with matching reflective padding; consumed by `PatchDataset.__getitem__`.

---
## Stage 4: PatchDataset — assemble one (input, ground-truth) sample

**Callable:** `PatchDataset(...).__getitem__(idx)`
**Input:** a single patch index `idx ∈ [0, number_of_patches)`.
**Output:** `(input_tensor, gt_params)` with `input_tensor : float32[C, p_h, p_w]` and `gt_params : float32[3K', p_h, p_w]`, where `C = input_config.total_channels(n_secondaries)` and `3K' = len(output_config.selected_indices(n_gaussians))`.

Each complex pass is converted to real channels by its `Representation` (here primary/secondaries → magnitude `|z|`, interferograms → phase `∠z`), concatenated in the fixed order primary → secondaries → interferograms → DEM. Ground-truth channels are gathered by `selected_indices`:

$$
\text{selected\_indices} = \big\{\,3g + i : g \in [0, K),\; i \in \text{role\_local\_indices}\,\big\}.
$$

> **What you should see:** `C` equals the analytic channel count — with this config
> `1·1 (primary mag) + S·1 (secondary mag) + S·1 (ifg phase) = 1 + 2S`. `gt_params` has
> `3·n_gaussians` channels (amp+mu+sigma per Gaussian). Interferogram-phase channels lie
> in `[-π, π]`; magnitude channels are non-negative. Both tensors are `float32` and
> finite. (Stage 4 builds the dataset **without** a normalizer or augmenter so the raw
> per-channel structure is visible.)

In [ ]:
raw_train_dataset = PatchDataset(
    inputs        = train_inputs,
    gt_parameters = train_parameters,
    grid          = train_patcher,
    input_config  = DATASET_CONFIG.input_config,
    output_config = DATASET_CONFIG.output_config,
    split_name    = "train",
    normalizer    = None,
    x_axis        = DATASET_CONFIG.x_axis,
    n_gaussians   = DATASET_CONFIG.n_gaussians,
    augmenter     = None,
    dem           = train_dem if DATASET_CONFIG.input_config.use_dem else None,
)

sample_patch_index            = 0
sample_input_tensor, sample_gt = raw_train_dataset[sample_patch_index]


In [ ]:
expected_input_channels = DATASET_CONFIG.input_config.total_channels(raw_train_dataset.n_secondaries)
expected_gt_channels    = len(DATASET_CONFIG.output_config.selected_indices(DATASET_CONFIG.n_gaussians))

print("Dataset length (patches)         :", len(raw_train_dataset))
print("Detected n_secondaries / n_slaves:", raw_train_dataset.n_secondaries, "/", raw_train_dataset.n_slaves)
print()
print("input_tensor shape / dtype       :", sample_input_tensor.shape, sample_input_tensor.dtype)
print("  expected input channels        :", expected_input_channels)
print("gt_params    shape / dtype       :", sample_gt.shape, sample_gt.dtype)
print("  expected gt channels (3K')     :", expected_gt_channels)
print("  selected gt indices            :", DATASET_CONFIG.output_config.selected_indices(DATASET_CONFIG.n_gaussians))
print()
print("input channel min / max per ch   :")
for channel_index in range(sample_input_tensor.shape[0]):
    channel = sample_input_tensor[channel_index]
    print(f"  ch {channel_index:>2}: min={channel.min():+.4f}  max={channel.max():+.4f}  mean={channel.mean():+.4f}")
print()
print("Finite input / gt                :", bool(np.isfinite(sample_input_tensor).all()), bool(np.isfinite(sample_gt).all()))


In [ ]:
n_input_channels = sample_input_tensor.shape[0]
n_columns        = min(n_input_channels, 5)

figure, axes = plt.subplots(2, n_columns, figsize=(2.6 * n_columns, 5.4))
axes = np.atleast_2d(axes)

for column in range(n_columns):
    channel_image = axes[0, column].imshow(sample_input_tensor[column], aspect="auto", cmap="cividis", origin="upper")
    axes[0, column].set_title(f"input ch {column}", fontsize=9)
    axes[0, column].set_xlabel("range (samples)")
    if column == 0:
        axes[0, column].set_ylabel("azimuth (lines)")
    figure.colorbar(channel_image, ax=axes[0, column], fraction=0.046)

    axes[1, column].hist(sample_input_tensor[column].ravel(), bins=40, color=PALETTE["primary"], edgecolor="white", linewidth=0.3)
    axes[1, column].set_title(f"input ch {column} values", fontsize=9)
    axes[1, column].set_xlabel("value")
    if column == 0:
        axes[1, column].set_ylabel("count")

figure.suptitle("Stage 4 — PatchDataset: per-channel input patch heatmaps and distributions", y=1.02)
figure.tight_layout()
figure.savefig(FIGURE_DIRECTORY / "stage_04_patchdataset.pdf")
plt.show()
print("Saved figure -> notebooks/figures/inspect_dataset_pipeline/stage_04_patchdataset.pdf")


In [ ]:
phase_channel_start    = DATASET_CONFIG.input_config.primary_channels_per_pass + raw_train_dataset.n_secondaries * DATASET_CONFIG.input_config.secondaries_channels_per_pass
interferogram_channels = sample_input_tensor[phase_channel_start:]
magnitude_channels     = sample_input_tensor[:phase_channel_start]

phase_in_range         = bool((interferogram_channels >= -np.pi - 1e-4).all() and (interferogram_channels <= np.pi + 1e-4).all()) if interferogram_channels.size else True
magnitude_non_negative = bool((magnitude_channels >= 0).all()) if magnitude_channels.size else True

checks = [
    ("input channels equal analytic total_channels", sample_input_tensor.shape[0] == expected_input_channels),
    ("gt channels equal 3K' selected indices",       sample_gt.shape[0] == expected_gt_channels),
    ("patch spatial size equals configured patch",   sample_input_tensor.shape[1:] == tuple(DATASET_CONFIG.patch.size)),
    ("input tensor dtype is float32",                sample_input_tensor.dtype == np.float32),
    ("gt tensor dtype is float32",                   sample_gt.dtype == np.float32),
    ("interferogram phase channels in [-pi, pi]",    phase_in_range),
    ("magnitude channels non-negative",              magnitude_non_negative),
    ("input and gt finite",                          bool(np.isfinite(sample_input_tensor).all() and np.isfinite(sample_gt).all())),
]
for description, condition in checks:
    print(f"[{'PASS' if condition else 'FAIL'}] {description}")


### Common Mistakes — Stage 4: PatchDataset

| Symptom | Likely Cause | How to Diagnose |
|---------|-------------|-----------------|
| Channel count off by `S` | `n_secondaries` miscounted from `input_layers` when only one of secondaries/ifgs is used | Print `dataset.n_secondaries` vs `(P-1)//2` |
| Channels in wrong order | primary/secondary/ifg/DEM offset bookkeeping in `_build_input_tensor` mis-stepped | Compare per-channel stats to expected magnitude vs phase signatures |
| Phase channel outside `[-π, π]` | magnitude representation applied where phase intended, or vice versa | Inspect interferogram channel range; check `interferograms_representation` |
| GT has wrong channels | `selected_indices` built from wrong `role_names` / `n_gaussians` | Print `output_config.selected_indices(n_gaussians)` |
| DEM channel missing/extra | `use_dem` true but `dem=None` passed (or the reverse) | Confirm `dem` argument matches `input_config.use_dem` |

**Passing to Stage 5:** `raw_train_dataset` — an un-normalized **train** `PatchDataset`; its samples are the population `StatsComputer` fits normalization statistics on.

---
## Stage 5: StatsComputer — fit normalization statistics on the TRAIN split only

**Callable:** `StatsComputer.compute(dataset=train_ds, params_path, input_config, output_config, n_slaves, n_gaussians, ...)`
**Input:** the **train** `PatchDataset` (for input channels) and the GT parameter cube path (for output channels).
**Output:** a `Stats` with per-channel `loc`, `scale`, strategy and `log1p` flags for both inputs and outputs.

Channels are grouped by slot-kind (e.g. all `pass/mag` share one fit, all `ifg/phase` share another). Each group's `(loc, scale)` follows its strategy: `min_max_p999` returns `(p_{0.1}, p_{99.9} - p_{0.1})`, `robust_iqr` returns `(\text{median}, \text{IQR})`, `zscore` returns `(\mu, \sigma)`, and `fixed_div_pi` returns `(0, \pi)`. `log1p` groups fit on `\log(1 + \max(x, 0))`.

> **What you should see:** the number of fitted input channels equals the train sample's
> channel count `C`; output channels equal `3·n_gaussians`. `scale` is strictly positive
> everywhere (the `max(·, 1e-8)` floor guarantees it). The `ifg/phase` group is fixed to
> `loc = 0`, `scale = π`. **Statistics are derived from the train split only** — no val /
> test pixels participate, so there is no normalization leakage.

In [ ]:
normalization_stats = StatsComputer.compute(
    dataset       = raw_train_dataset,
    params_path   = DATASET_CONFIG.parameters_path,
    logger        = INSPECTION_LOGGER,
    input_config  = DATASET_CONFIG.input_config,
    output_config = DATASET_CONFIG.output_config,
    n_slaves      = raw_train_dataset.n_slaves,
    n_gaussians   = DATASET_CONFIG.n_gaussians,
    num_workers   = 0,
    max_samples   = len(raw_train_dataset),
)


In [ ]:
input_stats  = normalization_stats.input_stats
output_stats = normalization_stats.output_stats

print("Input  channels fitted           :", input_stats.n_channels)
print("Output channels fitted           :", output_stats.n_channels)
print()
print("Input per-channel (name | method | log1p | loc | scale):")
for channel_index in range(input_stats.n_channels):
    strategy = input_stats.strategies[channel_index]
    print(f"  ch {channel_index:>2}: {input_stats.names[channel_index]:<14} {strategy.norm_method.value:<13} log1p={str(strategy.apply_log1p):<5} loc={input_stats.loc[channel_index]:+.5f}  scale={input_stats.scale[channel_index]:.5f}")
print()
print("Output per-channel (name | method | log1p | loc | scale):")
for channel_index in range(output_stats.n_channels):
    strategy = output_stats.strategies[channel_index]
    print(f"  ch {channel_index:>2}: {output_stats.names[channel_index]:<14} {strategy.norm_method.value:<13} log1p={str(strategy.apply_log1p):<5} loc={output_stats.loc[channel_index]:+.5f}  scale={output_stats.scale[channel_index]:.5f}")


In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12, 4.2))

input_positions = np.arange(input_stats.n_channels)
axes[0].bar(input_positions - 0.2, input_stats.loc,   width=0.4, color=PALETTE["primary"],   edgecolor="white", linewidth=0.4, label="loc")
axes[0].bar(input_positions + 0.2, input_stats.scale, width=0.4, color=PALETTE["secondary"], edgecolor="white", linewidth=0.4, label="scale")
axes[0].set_xticks(input_positions)
axes[0].set_xticklabels(input_stats.names, rotation=45, ha="right", fontsize=7)
axes[0].set_title("Stage 5 — StatsComputer: input channel loc / scale")
axes[0].set_xlabel("input channel")
axes[0].set_ylabel("statistic value")
axes[0].legend(loc="upper left", bbox_to_anchor=(1.0, 1.0), frameon=False)

output_positions = np.arange(output_stats.n_channels)
axes[1].bar(output_positions - 0.2, output_stats.loc,   width=0.4, color=PALETTE["primary"],   edgecolor="white", linewidth=0.4, label="loc")
axes[1].bar(output_positions + 0.2, output_stats.scale, width=0.4, color=PALETTE["secondary"], edgecolor="white", linewidth=0.4, label="scale")
axes[1].set_xticks(output_positions)
axes[1].set_xticklabels(output_stats.names, rotation=45, ha="right", fontsize=7)
axes[1].set_title("Stage 5 — StatsComputer: output channel loc / scale")
axes[1].set_xlabel("output channel")
axes[1].set_ylabel("statistic value")
axes[1].legend(loc="upper left", bbox_to_anchor=(1.0, 1.0), frameon=False)

figure.tight_layout()
figure.savefig(FIGURE_DIRECTORY / "stage_05_stats.pdf")
plt.show()
print("Saved figure -> notebooks/figures/inspect_dataset_pipeline/stage_05_stats.pdf")


In [ ]:
phase_groups_fixed = all(
    math.isclose(input_stats.loc[channel_index], 0.0) and math.isclose(input_stats.scale[channel_index], float(np.pi))
    for channel_index in range(input_stats.n_channels)
    if input_stats.strategies[channel_index].norm_method is NormMethod.FIXED_DIV_PI
)

checks = [
    ("input channel count equals sample channels", input_stats.n_channels == sample_input_tensor.shape[0]),
    ("output channel count equals 3K",             output_stats.n_channels == 3 * DATASET_CONFIG.n_gaussians),
    ("all input scales strictly positive",         all(scale > 0 for scale in input_stats.scale)),
    ("all output scales strictly positive",        all(scale > 0 for scale in output_stats.scale)),
    ("all input scales >= 1e-8 floor",             all(scale >= 1e-8 for scale in input_stats.scale)),
    ("fixed_div_pi groups use loc=0 scale=pi",     phase_groups_fixed),
    ("input loc / scale lengths agree",            len(input_stats.loc) == len(input_stats.scale)),
    ("strategies provided for every channel",      len(input_stats.strategies) == input_stats.n_channels),
]
for description, condition in checks:
    print(f"[{'PASS' if condition else 'FAIL'}] {description}")


### Common Mistakes — Stage 5: StatsComputer

| Symptom | Likely Cause | How to Diagnose |
|---------|-------------|-----------------|
| Validation loss artificially low | statistics fit on val/test or on the full cube (leakage) | Confirm `compute(dataset=train_ds, ...)` receives **only** the train dataset |
| `scale = 0` → divide-by-zero downstream | constant channel, `max(·, 1e-8)` floor removed | Print `min(scales)`; ensure floor present in `ChannelStrategy.fit` |
| Phase channels over-stretched | `fixed_div_pi` strategy not selected for `*/phase` slots | Check `loc=0, scale=π` for phase groups |
| Stats unstable run-to-run | `max_samples` subset RNG unseeded; here fixed to `default_rng(42)` | Re-run; loc/scale should be identical for a fixed subset |
| Wrong group keys vs channels | `channel_group_keys(n_slaves)` count `!=` tensor channels | The `assert` inside `compute_input_stats` guards this; print both counts |

**Passing to Stage 6:** `normalization_stats` — a `Stats` of train-fitted per-channel `loc`/`scale`/strategy; wrapped by a `Normalizer` and shared across all splits.

---
## Stage 6: Normalizer — standardize inputs and ground-truth parameters

**Callable:** `Normalizer(stats).normalize_input(tensor)` and `normalize_output(tensor)` (round-tripped via `denormalize_*`).
**Input:** a raw patch `input_tensor` / `gt_params`.
**Output:** standardized tensors centred near zero, with `log1p` applied to magnitude-like channels before centring.

Per channel the forward map is:

$$
x' = \begin{cases} \log(1 + \max(x, 0)) & \text{if log1p} \\ x & \text{otherwise} \end{cases},
\qquad
\hat{x} = \frac{x' - \text{loc}}{\text{scale}}
$$

and the inverse `denormalize` recovers `x = \text{scale}\cdot\hat{x} + \text{loc}` (then `expm1` for `log1p` channels). The pair forms an identity within numerical tolerance.

> **What you should see:** normalized channel values are roughly centred (median near 0)
> and on a comparable scale across channels; `min_max_p999` channels sit mostly in
> `[0, 1]` while `zscore` channels sit near unit variance. The denormalize → normalize
> round-trip reproduces the raw tensor to `< 1e-3` max absolute error. Phase channels map
> exactly by `/π` into `[-1, 1]`.

In [ ]:
normalizer = Normalizer(normalization_stats)

raw_input_for_norm      = sample_input_tensor.copy()
normalized_input_tensor = normalizer.normalize_input(raw_input_for_norm)
roundtrip_input_tensor  = normalizer.denormalize_input(normalized_input_tensor.copy())


In [ ]:
roundtrip_max_error = float(np.abs(roundtrip_input_tensor - raw_input_for_norm).max())

print("raw       min / max / mean       :", float(raw_input_for_norm.min()), float(raw_input_for_norm.max()), float(raw_input_for_norm.mean()))
print("normalized min / max / mean      :", float(normalized_input_tensor.min()), float(normalized_input_tensor.max()), float(normalized_input_tensor.mean()))
print("normalized dtype                 :", normalized_input_tensor.dtype)
print()
print("Per-channel normalized median:")
for channel_index in range(normalized_input_tensor.shape[0]):
    strategy = input_stats.strategies[channel_index]
    median   = float(np.median(normalized_input_tensor[channel_index]))
    print(f"  ch {channel_index:>2}: {input_stats.names[channel_index]:<14} {strategy.norm_method.value:<13} normalized median={median:+.4f}")
print()
print("Denormalize -> normalize round-trip max abs error:", roundtrip_max_error)


In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12, 4.4))

example_channel        = 0
raw_channel_values     = raw_input_for_norm[example_channel].ravel()
normalized_channel     = normalized_input_tensor[example_channel].ravel()

axes[0].hist(raw_channel_values, bins=45, color=PALETTE["before"], edgecolor="white", linewidth=0.3, label="raw", alpha=0.85)
axes_twin = axes[0].twiny()
axes_twin.hist(normalized_channel, bins=45, color=PALETTE["after"], edgecolor="white", linewidth=0.3, label="normalized", alpha=0.55)
axes[0].set_title(f"Stage 6 — Normalizer: channel {example_channel} before / after")
axes[0].set_xlabel("raw value")
axes_twin.set_xlabel("normalized value")
axes[0].set_ylabel("count")

channel_medians = [float(np.median(normalized_input_tensor[channel_index])) for channel_index in range(normalized_input_tensor.shape[0])]
channel_iqrs    = [float(np.percentile(normalized_input_tensor[channel_index], 75) - np.percentile(normalized_input_tensor[channel_index], 25)) for channel_index in range(normalized_input_tensor.shape[0])]
positions       = np.arange(len(channel_medians))
axes[1].errorbar(positions, channel_medians, yerr=np.array(channel_iqrs) / 2.0, fmt="o", color=PALETTE["primary"], ecolor=PALETTE["secondary"], capsize=3, linewidth=1.2)
axes[1].axhline(0.0, color=PALETTE["neutral"], linewidth=0.8, linestyle="--")
axes[1].set_xticks(positions)
axes[1].set_xticklabels(input_stats.names, rotation=45, ha="right", fontsize=7)
axes[1].set_title("Stage 6 — Normalizer: normalized median +/- IQR per channel")
axes[1].set_xlabel("input channel")
axes[1].set_ylabel("normalized value")

figure.tight_layout()
figure.savefig(FIGURE_DIRECTORY / "stage_06_normalizer.pdf")
plt.show()
print("Saved figure -> notebooks/figures/inspect_dataset_pipeline/stage_06_normalizer.pdf")


In [ ]:
phase_channel_indices = [channel_index for channel_index in range(input_stats.n_channels) if input_stats.strategies[channel_index].norm_method is NormMethod.FIXED_DIV_PI]
phase_within_unit     = all(bool((np.abs(normalized_input_tensor[channel_index]) <= 1.0 + 1e-4).all()) for channel_index in phase_channel_indices) if phase_channel_indices else True

checks = [
    ("normalized output is float32",               normalized_input_tensor.dtype == np.float32),
    ("normalized shape equals input shape",        normalized_input_tensor.shape == raw_input_for_norm.shape),
    ("normalized tensor finite",                   bool(np.isfinite(normalized_input_tensor).all())),
    ("round-trip max abs error < 1e-3",            roundtrip_max_error < 1e-3),
    ("phase channels map into [-1, 1]",            phase_within_unit),
    ("normalized global mean within [-3, 3]",      -3.0 <= float(normalized_input_tensor.mean()) <= 3.0),
]
for description, condition in checks:
    print(f"[{'PASS' if condition else 'FAIL'}] {description}")


### Common Mistakes — Stage 6: Normalizer

| Symptom | Likely Cause | How to Diagnose |
|---------|-------------|-----------------|
| Round-trip does not recover raw | `log1p`/`expm1` mismatch, or `loc`/`scale` applied in wrong order | Diff `denormalize(normalize(x))` against `x`; should be `< 1e-3` |
| Exploding values after denorm | `expm1` without the `clamp(max=15)` guard on log1p channels | Check the `x.clamp(max=15.0)` branch; inspect channel max |
| Channels not comparable in scale | a channel's strategy/loc/scale not aligned to its slot-kind | Compare per-channel normalized median/IQR; phase must be `/π` |
| Negative magnitudes after log1p | `np.maximum(x, 0)` clamp removed before `log1p` | Inspect raw magnitude channel min before normalization |
| Reshape broadcast wrong | 3-D vs 4-D shape branch `(-1,1,1)` vs `(1,-1,1,1)` mismatch | Print `tensor.ndim`; confirm the chosen reshape |

**Passing to Stage 7:** `normalizer` — the train-fitted standardizer; attached to the train dataset and reused unchanged for val / test.

---
## Stage 7: SpatialAugmenter — train-only flips, 90° rotations, and noise

**Callable:** `SpatialAugmenter(augmentation_config, logger)(input_tensor, gt_params)`
**Input:** one `(input_tensor, gt_params)` pair (applied **only** when `split_name == "train"`).
**Output:** a spatially transformed and optionally noised pair; the **same** geometric transform is applied to input and ground truth.

With independent Bernoulli draws the augmenter may horizontally flip (`p_flip_h`), vertically flip (`p_flip_v`), rotate by `k·90°` (`p_rot90`), and add Gaussian noise `\mathcal{N}(0, \sigma_{\text{noise}}^2)` (`p_noise`) to the input only:

$$
\tilde{x} = R_k\!\big(F_v F_h\, x\big) + n, \quad n \sim \mathcal{N}(0, \sigma_{\text{noise}}^2), \qquad
\tilde{y} = R_k\!\big(F_v F_h\, y\big).
$$

> **What you should see:** the geometric transform applied to the input matches the one
> applied to the ground truth (verified by forcing a deterministic flip and comparing
> against a manual `numpy` flip). Output shapes are preserved (rotation here uses square
> patches). Noise, when applied, perturbs the input but **never** the ground truth, with
> an empirical standard deviation close to `noise_std`.

In [ ]:
augmenter = SpatialAugmenter(DATASET_CONFIG.augmentation, logger=INSPECTION_LOGGER)
augmenter._rng = np.random.default_rng(RANDOM_SEED)

augmentation_input  = normalized_input_tensor.copy()
augmentation_gt     = sample_gt.copy()
augmented_input, augmented_gt = augmenter(augmentation_input.copy(), augmentation_gt.copy())


In [ ]:
deterministic_config    = AugmentationConfig(p_flip_h=1.0, p_flip_v=0.0, p_rot90=0.0, noise_std=0.0, p_noise=0.0)
deterministic_augmenter = SpatialAugmenter(deterministic_config, logger=INSPECTION_LOGGER)
forced_input, forced_gt = deterministic_augmenter(normalized_input_tensor.copy(), sample_gt.copy())

manual_flip_input = normalized_input_tensor[..., ::-1]
manual_flip_gt    = sample_gt[..., ::-1]

print("augmented input shape / dtype    :", augmented_input.shape, augmented_input.dtype)
print("augmented gt    shape / dtype    :", augmented_gt.shape, augmented_gt.dtype)
print()
print("Forced horizontal-flip equivalence (vs manual numpy flip):")
print("  input max abs diff             :", float(np.abs(forced_input - manual_flip_input).max()))
print("  gt    max abs diff             :", float(np.abs(forced_gt - manual_flip_gt).max()))
print()
print("Empirical noise std (augmented input - geometric-only baseline):")
noise_only_config    = AugmentationConfig(p_flip_h=0.0, p_flip_v=0.0, p_rot90=0.0, noise_std=DATASET_CONFIG.augmentation.noise_std, p_noise=1.0)
noise_only_augmenter = SpatialAugmenter(noise_only_config, logger=INSPECTION_LOGGER)
noised_input, noised_gt = noise_only_augmenter(normalized_input_tensor.copy(), sample_gt.copy())
print("  measured input perturbation std:", float((noised_input - normalized_input_tensor).std()))
print("  configured noise_std           :", DATASET_CONFIG.augmentation.noise_std)
print("  gt untouched by noise (max diff):", float(np.abs(noised_gt - sample_gt).max()))


In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(12, 4.0))

original_image = axes[0].imshow(normalized_input_tensor[0], aspect="auto", cmap="cividis", origin="upper")
axes[0].set_title("Stage 7 — Augmenter: original ch 0")
axes[0].set_xlabel("range (samples)")
axes[0].set_ylabel("azimuth (lines)")
figure.colorbar(original_image, ax=axes[0], fraction=0.046)

flipped_image = axes[1].imshow(forced_input[0], aspect="auto", cmap="cividis", origin="upper")
axes[1].set_title("Stage 7 — Augmenter: forced H-flip ch 0")
axes[1].set_xlabel("range (samples)")
figure.colorbar(flipped_image, ax=axes[1], fraction=0.046)

noise_field   = noised_input[0] - normalized_input_tensor[0]
noise_image   = axes[2].imshow(noise_field, aspect="auto", cmap="coolwarm", origin="upper")
axes[2].set_title("Stage 7 — Augmenter: additive noise field ch 0")
axes[2].set_xlabel("range (samples)")
figure.colorbar(noise_image, ax=axes[2], fraction=0.046, label="noise (value)")

figure.tight_layout()
figure.savefig(FIGURE_DIRECTORY / "stage_07_augmentation.pdf")
plt.show()
print("Saved figure -> notebooks/figures/inspect_dataset_pipeline/stage_07_augmentation.pdf")


In [ ]:
measured_noise_std = float((noised_input - normalized_input_tensor).std())

checks = [
    ("augmented input shape preserved",            augmented_input.shape == normalized_input_tensor.shape),
    ("augmented gt shape preserved",               augmented_gt.shape == sample_gt.shape),
    ("forced flip matches manual input flip",      float(np.abs(forced_input - manual_flip_input).max()) < 1e-6),
    ("forced flip matches manual gt flip",         float(np.abs(forced_gt - manual_flip_gt).max()) < 1e-6),
    ("noise leaves ground truth untouched",        float(np.abs(noised_gt - sample_gt).max()) == 0.0),
    ("measured noise std near configured std",     abs(measured_noise_std - DATASET_CONFIG.augmentation.noise_std) < 0.5 * DATASET_CONFIG.augmentation.noise_std + 1e-4),
    ("augmented tensors finite",                   bool(np.isfinite(augmented_input).all() and np.isfinite(augmented_gt).all())),
]
for description, condition in checks:
    print(f"[{'PASS' if condition else 'FAIL'}] {description}")


### Common Mistakes — Stage 7: SpatialAugmenter

| Symptom | Likely Cause | How to Diagnose |
|---------|-------------|-----------------|
| Input and GT geometrically misaligned | flip/rotate applied to only one tensor, or with different `k` | Force a deterministic flip; diff input and GT against the same manual flip |
| Augmentation leaks into val/test | augmenter invoked when `split_name != "train"` | The `__getitem__` guard checks `split_name == "train"`; verify val sample is unaugmented |
| Non-reproducible epochs | `default_rng()` unseeded; here re-seeded explicitly for inspection | Note production seeds per-worker; set `_rng` to inspect deterministically |
| Noise added to ground truth | noise applied to the wrong tensor in `__call__` | Confirm GT max-diff under noise-only config is exactly 0 |
| Rotation breaks non-square patches | `rot90` on a non-square patch changes shape, corrupting collation | Keep `p_rot90=0` unless `p_h == p_w` |

**Passing to Stage 8:** `normalizer` (still the train-fitted one) — now attached to the train dataset and handed to the val / test dataset constructors so all splits share identical statistics.

---
## Stage 8: Build val and test datasets — shared train-fitted normalizer

**Callable:** `DatasetPipeline._build_dataset("val", normalizer)` and `_build_dataset("test", normalizer)`.
**Input:** the val / test `CropRegion`s and the **train-fitted** `normalizer`.
**Output:** two `PatchDataset`s whose patches are normalized with the *same* `loc`/`scale` as the train split, each with its own patch grid sized to its region.

Reusing the train normalizer is the central anti-leakage invariant:

$$
\text{normalizer}_{\text{train}} = \text{normalizer}_{\text{val}} = \text{normalizer}_{\text{test}}.
$$

> **What you should see:** all three datasets expose the same channel counts (`C` input,
> `3K'` GT) and the same patch spatial size. Their lengths follow each split's azimuth
> extent: the train split (70%) yields the most patches, val and test (~15% each) fewer.
> The val and test datasets carry the **identical** `normalizer` object reference used by
> the train dataset — confirmed by object identity, not value equality.

In [ ]:
dataset_pipeline                  = DatasetPipeline(config=DATASET_CONFIG, training_run_directory=TRAINING_RUN_DIRECTORY, logger=INSPECTION_LOGGER)

train_dataset_with_norm, train_patcher_built = dataset_pipeline._build_dataset("train")
train_dataset_with_norm.normalizer           = normalizer

val_dataset,  val_patcher_built  = dataset_pipeline._build_dataset("val",  normalizer=normalizer)
test_dataset, test_patcher_built = dataset_pipeline._build_dataset("test", normalizer=normalizer)


In [ ]:
split_datasets = {"train": train_dataset_with_norm, "val": val_dataset, "test": test_dataset}

print("Split sizes and shared normalizer:")
for split_name, dataset in split_datasets.items():
    print(f"  {split_name:<5}: patches={len(dataset):>4}  in_channels={dataset.input_channels}  gt_channels={dataset.gt_channels}  normalizer_is_train={dataset.normalizer is normalizer}")
print()
val_sample_input, val_sample_gt = val_dataset[0]
print("val sample input shape / dtype   :", val_sample_input.shape, val_sample_input.dtype)
print("val sample gt    shape / dtype   :", val_sample_gt.shape, val_sample_gt.dtype)
print()
total_patches = sum(len(dataset) for dataset in split_datasets.values())
print("Total patches across splits      :", total_patches)
print("Split fractions (patches)        :", {name: round(len(dataset) / total_patches, 3) for name, dataset in split_datasets.items()})


In [ ]:
split_names   = list(split_datasets.keys())
split_lengths = [len(split_datasets[name]) for name in split_names]
split_azimuth = [dict(DATASET_CONFIG.split_regions.items())[name].azimuth_size for name in split_names]

figure, axes = plt.subplots(1, 2, figsize=(11, 4.0))

bars = axes[0].bar(split_names, split_lengths, color=[PALETTE["primary"], PALETTE["secondary"], PALETTE["accent"]], edgecolor="white", linewidth=0.6)
for bar, length in zip(bars, split_lengths):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height(), str(length), ha="center", va="bottom", fontsize=9)
axes[0].set_title("Stage 8 — Splits: patch count per split")
axes[0].set_xlabel("split")
axes[0].set_ylabel("number of patches")

axes[1].scatter(split_azimuth, split_lengths, color=PALETTE["primary"], s=60, zorder=3)
for name, azimuth, length in zip(split_names, split_azimuth, split_lengths):
    axes[1].annotate(name, (azimuth, length), textcoords="offset points", xytext=(6, 4), fontsize=9)
axes[1].set_title("Stage 8 — Splits: patches vs azimuth extent")
axes[1].set_xlabel("split azimuth extent (lines)")
axes[1].set_ylabel("number of patches")

figure.tight_layout()
figure.savefig(FIGURE_DIRECTORY / "stage_08_splits.pdf")
plt.show()
print("Saved figure -> notebooks/figures/inspect_dataset_pipeline/stage_08_splits.pdf")


In [ ]:
train_azimuth_extent = dict(DATASET_CONFIG.split_regions.items())["train"].azimuth_size
val_azimuth_extent   = dict(DATASET_CONFIG.split_regions.items())["val"].azimuth_size

checks = [
    ("val normalizer is the train normalizer",     val_dataset.normalizer is normalizer),
    ("test normalizer is the train normalizer",    test_dataset.normalizer is normalizer),
    ("all splits share input channel count",       len({dataset.input_channels for dataset in split_datasets.values()}) == 1),
    ("all splits share gt channel count",          len({dataset.gt_channels for dataset in split_datasets.values()}) == 1),
    ("train has the most patches",                 len(train_dataset_with_norm) >= len(val_dataset) and len(train_dataset_with_norm) >= len(test_dataset)),
    ("patch counts scale with azimuth extent",     (len(train_dataset_with_norm) >= len(val_dataset)) == (train_azimuth_extent >= val_azimuth_extent)),
    ("val sample channels match train",            val_sample_input.shape[0] == train_dataset_with_norm.input_channels),
    ("every split non-empty",                      all(len(dataset) > 0 for dataset in split_datasets.values())),
]
for description, condition in checks:
    print(f"[{'PASS' if condition else 'FAIL'}] {description}")


### Common Mistakes — Stage 8: Split datasets

| Symptom | Likely Cause | How to Diagnose |
|---------|-------------|-----------------|
| Val/test re-fit their own statistics | a fresh `Normalizer` built per split instead of passing the train one | Assert `val.normalizer is train.normalizer` (identity, not equality) |
| Channel count differs across splits | different `input_config` / `n_gaussians` per split | Compare `input_channels` / `gt_channels` across splits |
| Train augmentation bleeds to val | augmenter attached but split-guard removed | Confirm a val sample equals its un-augmented patch |
| Split sizes implausible | `from_ratios` boundaries collapsed (one split empty) | Print per-split `azimuth_size` and patch counts |
| Spatial leakage across splits | azimuth windows overlap | Verify `train.azimuth_end == val.azimuth_start` etc. |

**Passing to Stage 9:** `train_dataset_with_norm`, `val_dataset`, `test_dataset` — three normalized `PatchDataset`s ready to be wrapped in `DataLoader`s.

---
## Stage 9: Loader — assemble train / val / test DataLoaders

**Callable:** `Loader.build(train_dataset, val_dataset, test_dataset, batch_size, num_workers, pin_memory, shuffle_train, logger)`
**Input:** the three datasets plus loader config.
**Output:** three `DataLoader`s — train shuffled with `drop_last=True`, val / test ordered with `drop_last=False`.

The number of train batches drops the final partial batch:

$$
\text{train\_batches} = \left\lfloor \frac{|\text{train}|}{B} \right\rfloor, \qquad
\text{val\_batches} = \left\lceil \frac{|\text{val}|}{B} \right\rceil.
$$

> **What you should see:** a train batch has shape `(B, C, p_h, p_w)` for inputs and
> `(B, 3K', p_h, p_w)` for ground truth, all `float32` and finite. `len(train_loader) ==
> len(train_dataset) // batch_size` (last partial batch dropped); `len(val_loader) ==
> ceil(len(val_dataset) / batch_size)`. The train loader shuffles (`shuffle=True`); val /
> test preserve order.

In [ ]:
train_loader, val_loader, test_loader = Loader.build(
    train_dataset = train_dataset_with_norm,
    val_dataset   = val_dataset,
    test_dataset  = test_dataset,
    batch_size    = DATASET_CONFIG.batch_size,
    num_workers   = DATASET_CONFIG.num_workers,
    pin_memory    = DATASET_CONFIG.pin_memory,
    shuffle_train = DATASET_CONFIG.shuffle_train,
    logger        = INSPECTION_LOGGER,
)

first_input_batch, first_gt_batch = next(iter(train_loader))


In [ ]:
batch_size = DATASET_CONFIG.batch_size

print("Train batches                    :", len(train_loader), " expected floor(|train|/B):", len(train_dataset_with_norm) // batch_size)
print("Val   batches                    :", len(val_loader),   " expected ceil(|val|/B)   :", math.ceil(len(val_dataset) / batch_size))
print("Test  batches                    :", len(test_loader),  " expected ceil(|test|/B)  :", math.ceil(len(test_dataset) / batch_size))
print()
print("First train input batch shape    :", tuple(first_input_batch.shape), first_input_batch.dtype)
print("First train gt    batch shape    :", tuple(first_gt_batch.shape), first_gt_batch.dtype)
print()
print("Batch input min / max / mean     :", float(first_input_batch.min()), float(first_input_batch.max()), float(first_input_batch.mean()))
print("Batch gt    min / max / mean     :", float(first_gt_batch.min()), float(first_gt_batch.max()), float(first_gt_batch.mean()))
print("Batch finite (input / gt)        :", bool(torch.isfinite(first_input_batch).all()), bool(torch.isfinite(first_gt_batch).all()))
print()
print("train shuffle / drop_last        :", train_loader.sampler.__class__.__name__, "/", train_loader.drop_last)
print("val   shuffle / drop_last        :", val_loader.sampler.__class__.__name__, "/", val_loader.drop_last)


In [ ]:
batch_input_numpy = first_input_batch.numpy()

figure, axes = plt.subplots(1, 2, figsize=(11, 4.2))

axes[0].hist(batch_input_numpy.ravel(), bins=60, color=PALETTE["primary"], edgecolor="white", linewidth=0.3)
axes[0].axvline(float(batch_input_numpy.mean()), color=PALETTE["accent"], linewidth=1.4, linestyle="--", label=f"mean={batch_input_numpy.mean():.3f}")
axes[0].axvline(float(batch_input_numpy.mean() + batch_input_numpy.std()), color=PALETTE["secondary"], linewidth=1.0, linestyle=":", label=f"+/-1 std={batch_input_numpy.std():.3f}")
axes[0].axvline(float(batch_input_numpy.mean() - batch_input_numpy.std()), color=PALETTE["secondary"], linewidth=1.0, linestyle=":")
axes[0].set_title("Stage 9 — Loader: train batch input distribution")
axes[0].set_xlabel("normalized value")
axes[0].set_ylabel("count")
axes[0].legend(frameon=False)

per_channel_mean = batch_input_numpy.mean(axis=(0, 2, 3))
axes[1].bar(range(len(per_channel_mean)), per_channel_mean, color=PALETTE["secondary"], edgecolor="white", linewidth=0.5)
axes[1].axhline(0.0, color=PALETTE["neutral"], linewidth=0.8, linestyle="--")
axes[1].set_title("Stage 9 — Loader: batch mean per input channel")
axes[1].set_xlabel("input channel")
axes[1].set_ylabel("mean normalized value")

figure.tight_layout()
figure.savefig(FIGURE_DIRECTORY / "stage_09_loader.pdf")
plt.show()
print("Saved figure -> notebooks/figures/inspect_dataset_pipeline/stage_09_loader.pdf")


In [ ]:
checks = [
    ("train batch is 4-D (B, C, H, W)",            first_input_batch.ndim == 4),
    ("train batch B equals batch_size",            first_input_batch.shape[0] == batch_size),
    ("train batch channels equal in_channels",     first_input_batch.shape[1] == train_dataset_with_norm.input_channels),
    ("train batch spatial equals patch size",      tuple(first_input_batch.shape[2:]) == tuple(DATASET_CONFIG.patch.size)),
    ("gt batch channels equal gt_channels",        first_gt_batch.shape[1] == train_dataset_with_norm.gt_channels),
    ("train batches = floor(|train|/B) (drop_last)", len(train_loader) == len(train_dataset_with_norm) // batch_size),
    ("val batches = ceil(|val|/B)",                len(val_loader) == math.ceil(len(val_dataset) / batch_size)),
    ("train loader drops last partial batch",      train_loader.drop_last is True),
    ("val loader keeps last partial batch",        val_loader.drop_last is False),
    ("batch tensors finite",                       bool(torch.isfinite(first_input_batch).all() and torch.isfinite(first_gt_batch).all())),
]
for description, condition in checks:
    print(f"[{'PASS' if condition else 'FAIL'}] {description}")


### Common Mistakes — Stage 9: Loader

| Symptom | Likely Cause | How to Diagnose |
|---------|-------------|-----------------|
| Val/test silently drop samples | `drop_last=True` set on eval loaders | Assert `val_loader.drop_last is False` |
| Train sees data in fixed order | `shuffle_train=False` or sampler overridden | Check `train_loader.sampler` is `RandomSampler` |
| Collation error / ragged batch | per-sample channel or spatial size varies | Print `first_input_batch.shape`; all samples must match |
| Worker non-determinism | `num_workers > 0` without seeded `worker_init_fn` | For reproducible inspection use `num_workers=0` |
| Batch all-NaN | upstream normalization produced Inf (zero scale) | Trace back to Stage 5 scales; check `torch.isfinite(batch)` |

**Passing to Stage 10:** `train_loader`, `val_loader`, `test_loader` — the consumable batched iterators; the pipeline finally persists metadata describing how they were built.

---
## Stage 10: MetadataWriter — persist crop, patch, and dataset config

**Callable:** `MetadataWriter.save_dataset_configuration(config)`, `save_crop_metadata(global_crop, splits)`, `save_patch_metadata(grids)`.
**Input:** the `DatasetConfiguration`, the global crop + per-split crops, and the per-split `GridInfo`s.
**Output:** three JSON files under `<training_run>/meta/`: `dataset_creation_config.json`, `crop.json`, `patch.json` — the reproducibility record for the run.

These files let inference and stitching reconstruct the exact patch grid and crop windows, so the persisted `n_v`, `n_h`, pad budget, and split crops must match the in-memory `GridInfo` and `CropRegion`s used during training.

> **What you should see:** all three JSON files exist and parse. `crop.json` round-trips
> the global crop and every split crop. `patch.json` contains a per-split grid whose
> `number_of_patches` equals the corresponding dataset length. `dataset_creation_config.json`
> records the patch size, stride, and input/output config used to build the datasets.

In [ ]:
metadata_writer = MetadataWriter(run_directory=TRAINING_RUN_DIRECTORY, logger=INSPECTION_LOGGER)

dataset_config_path = metadata_writer.save_dataset_configuration(DATASET_CONFIG)
crop_metadata_path  = metadata_writer.save_crop_metadata(global_crop=layout.global_crop, splits=dict(DATASET_CONFIG.split_regions.items()))
patch_metadata_path = metadata_writer.save_patch_metadata({
    "train" : train_patcher_built.grid,
    "val"   : val_patcher_built.grid,
    "test"  : test_patcher_built.grid,
})


In [ ]:
with open(crop_metadata_path, "r", encoding="utf-8") as handle:
    crop_metadata = json.load(handle)
with open(patch_metadata_path, "r", encoding="utf-8") as handle:
    patch_metadata = json.load(handle)
with open(dataset_config_path, "r", encoding="utf-8") as handle:
    dataset_config_metadata = json.load(handle)

print("Written metadata files:")
for written_path in [dataset_config_path, crop_metadata_path, patch_metadata_path]:
    print(f"  exists={written_path.exists()}  {written_path.name}")
print()
print("crop.json global_crop            :", crop_metadata["global_crop"])
print("crop.json split keys             :", list(crop_metadata["splits"].keys()))
print()
print("patch.json per-split grid summary:")
for split_name, grid_payload in patch_metadata.items():
    print(f"  {split_name:<5}: n_v={grid_payload['n_v']} n_h={grid_payload['n_h']} -> {grid_payload['number_of_patches']} patches  patch_size={grid_payload['patch_size']} stride={grid_payload['stride']}")
print()
print("dataset_creation_config patch    :", dataset_config_metadata.get("patch"))
print("dataset_creation_config in cfg   :", dataset_config_metadata.get("input_config"))


In [ ]:
split_names_meta = ["train", "val", "test"]
patches_in_meta  = [patch_metadata[name]["number_of_patches"] for name in split_names_meta]
patches_in_memory = [len(split_datasets[name]) for name in split_names_meta]

figure, axes = plt.subplots(1, 1, figsize=(7.5, 4.2))

bar_positions = np.arange(len(split_names_meta))
axes.bar(bar_positions - 0.2, patches_in_memory, width=0.4, color=PALETTE["primary"],   edgecolor="white", linewidth=0.5, label="in-memory dataset")
axes.bar(bar_positions + 0.2, patches_in_meta,   width=0.4, color=PALETTE["secondary"], edgecolor="white", linewidth=0.5, label="persisted patch.json")
axes.set_xticks(bar_positions)
axes.set_xticklabels(split_names_meta)
axes.set_title("Stage 10 — MetadataWriter: persisted vs in-memory patch counts")
axes.set_xlabel("split")
axes.set_ylabel("number of patches")
axes.legend(frameon=False)

figure.tight_layout()
figure.savefig(FIGURE_DIRECTORY / "stage_10_metadata.pdf")
plt.show()
print("Saved figure -> notebooks/figures/inspect_dataset_pipeline/stage_10_metadata.pdf")


In [ ]:
global_crop_roundtrips = crop_metadata["global_crop"] == list(layout.global_crop.as_tuple())
splits_roundtrip       = all(crop_metadata["splits"][name] == list(region.as_tuple()) for name, region in DATASET_CONFIG.split_regions.items())
patch_counts_match     = all(patch_metadata[name]["number_of_patches"] == len(split_datasets[name]) for name in split_names_meta)

checks = [
    ("dataset_creation_config.json exists",         dataset_config_path.exists()),
    ("crop.json exists",                            crop_metadata_path.exists()),
    ("patch.json exists",                           patch_metadata_path.exists()),
    ("crop.json global crop round-trips",           global_crop_roundtrips),
    ("crop.json every split crop round-trips",       splits_roundtrip),
    ("patch.json counts match dataset lengths",     patch_counts_match),
    ("patch.json patch size matches config",        tuple(patch_metadata["train"]["patch_size"]) == tuple(DATASET_CONFIG.patch.size)),
    ("patch.json stride matches config",            patch_metadata["train"]["stride"] == DATASET_CONFIG.patch.stride),
]
for description, condition in checks:
    print(f"[{'PASS' if condition else 'FAIL'}] {description}")


### Common Mistakes — Stage 10: MetadataWriter

| Symptom | Likely Cause | How to Diagnose |
|---------|-------------|-----------------|
| Inference stitches with the wrong grid | `patch.json` written from a stale `GridInfo` (different stride/pad) | Diff `patch.json` against the live `GridInfo.as_dict()` |
| Crop windows shifted at inference | `crop.json` global/split crops not matching the arrays used | Round-trip `crop.json` against `layout.global_crop` and split crops |
| Config JSON missing nested fields | `asdict` drops a non-dataclass field, or `x_axis` left in | Confirm `input_config` / `output_config` keys present; `x_axis` popped |
| `meta/` directory absent | `MetadataWriter.__init__` did not `mkdir` the run dir | Check `metadata_writer.metadata_directory.exists()` |
| Patch counts disagree with datasets | grids passed for the wrong split (val grid under `train` key) | Compare persisted counts to `len(dataset)` per split |

**Passing to end-to-end summary:** every stage's output has been validated in isolation; the final group runs the full orchestrator and cross-checks against this assembled view.

---
## End-to-End Summary — full `DatasetPipeline.run()` vs assembled stages

This final group runs the complete orchestrator `DatasetPipeline.run()` on the same
synthetic fixture and confirms its outputs are consistent with the step-by-step
assembled view above: the three returned `DataLoader`s carry the expected batch shapes,
the returned datasets share the train-fitted normalizer, and the persisted metadata
matches. A structured table then reports every intermediate output's shape, dtype, and
key statistics in one place.

In [ ]:
run_train_loader, run_val_loader, run_test_loader, run_datasets = dataset_pipeline.run()

run_input_batch, run_gt_batch = next(iter(run_train_loader))


In [ ]:
run_train_dataset = run_datasets["train"]
run_val_dataset   = run_datasets["val"]
run_test_dataset  = run_datasets["test"]

print("Orchestrator vs assembled:")
print(f"  train batch shape (run)        : {tuple(run_input_batch.shape)}")
print(f"  train batch shape (assembled)  : {tuple(first_input_batch.shape)}")
print(f"  gt    batch shape (run)        : {tuple(run_gt_batch.shape)}")
print()
print("  run train normalizer shared val:", run_val_dataset.normalizer is run_train_dataset.normalizer)
print("  run train normalizer shared test:", run_test_dataset.normalizer is run_train_dataset.normalizer)
print()
print("  run loader batch counts (train/val/test):", len(run_train_loader), len(run_val_loader), len(run_test_loader))


In [ ]:
summary_rows = [
    ("Stage 1  Layout global crop",     str(layout.global_crop.as_tuple()),                     "CropRegion",  f"az={layout.global_crop.azimuth_size} rg={layout.global_crop.range_size}"),
    ("Stage 2  Cropper train inputs",   str(train_inputs.shape),                                str(train_inputs.dtype),  f"|mag| mean={np.abs(train_inputs).mean():.3f}"),
    ("Stage 3  Patcher train grid",     f"{train_grid.n_v}x{train_grid.n_h}",                   "GridInfo",    f"{train_grid.number_of_patches} patches"),
    ("Stage 4  PatchDataset sample in", str(sample_input_tensor.shape),                         str(sample_input_tensor.dtype), f"mean={sample_input_tensor.mean():.3f}"),
    ("Stage 4  PatchDataset sample gt", str(sample_gt.shape),                                   str(sample_gt.dtype), f"mean={sample_gt.mean():.3f}"),
    ("Stage 5  Stats input channels",   str(input_stats.n_channels),                            "ChannelStats", f"scale_min={min(input_stats.scale):.3f}"),
    ("Stage 6  Normalized sample in",   str(normalized_input_tensor.shape),                     str(normalized_input_tensor.dtype), f"mean={normalized_input_tensor.mean():.3f}"),
    ("Stage 7  Augmented sample in",    str(augmented_input.shape),                             str(augmented_input.dtype), f"noise_std~={measured_noise_std:.4f}"),
    ("Stage 8  Split patch counts",     str({name: len(dataset) for name, dataset in split_datasets.items()}), "dict", "shared normalizer"),
    ("Stage 9  Train batch",            str(tuple(first_input_batch.shape)),                    str(first_input_batch.dtype), f"mean={float(first_input_batch.mean()):.3f}"),
    ("Stage 10 Persisted metadata",     "crop.json/patch.json/config.json",                     "JSON",        f"train patches={patch_metadata['train']['number_of_patches']}"),
]

header = f"{'Stage / artifact':<34}{'shape / value':<28}{'type':<16}{'key statistic'}"
print(header)
print("-" * len(header))
for label, shape_value, type_value, statistic in summary_rows:
    print(f"{label:<34}{shape_value:<28}{type_value:<16}{statistic}")


In [ ]:
final_checks = [
    ("run train batch matches assembled batch shape",  tuple(run_input_batch.shape) == tuple(first_input_batch.shape)),
    ("run gt batch matches assembled gt shape",        tuple(run_gt_batch.shape) == tuple(first_gt_batch.shape)),
    ("run datasets share one normalizer",              run_val_dataset.normalizer is run_train_dataset.normalizer and run_test_dataset.normalizer is run_train_dataset.normalizer),
    ("run train batch finite",                         bool(torch.isfinite(run_input_batch).all() and torch.isfinite(run_gt_batch).all())),
    ("run produced all three loaders",                 all(loader is not None for loader in [run_train_loader, run_val_loader, run_test_loader])),
    ("run datasets cover train/val/test",              set(run_datasets.keys()) == {"train", "val", "test"}),
]
for description, condition in final_checks:
    print(f"[{'PASS' if condition else 'FAIL'}] {description}")


In [ ]:
shutil.rmtree(WORKING_RUN_ROOT, ignore_errors=True)
print("Removed synthetic working directory:", WORKING_RUN_ROOT)
print("Inspection complete — all ten dataset-pipeline stages exercised end to end.")
